In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/global/cfs/cdirs/desi/spectro/redux/fuji/exposures-fuji.fits', ext='EXPOSURES'))
print(len(cat), len(np.unique(cat['EXPID'])))

2470 2470


In [4]:
explist = Table.read('/global/u2/r/rongpu/notebooks/sv/everest/rr_commands/low_speed_coadds.txt', format='ascii.commented_header')
explist

TILEID,NEXP,EFFTIME,SPEED_DARK,EFFTIMES,SPEED_DARKS,EXPIDS
int64,int64,int64,float64,str21,str29,str35
80605,5,1052,0.26,"240,189,198,238,188","0.40,0.23,0.26,0.23,0.17","67972,67974,67987,73704,73705"
80605,4,1093,0.46,"189,380,199,325","0.26,0.56,0.31,0.55","67973,67975,71594,73702"
80608,6,1119,0.34,"370,116,270,225,52,86","0.48,0.10,0.40,0.29,0.05,0.10","67771,68013,68491,68492,69625,69626"
80608,4,1135,0.44,"370,116,424,225","0.48,0.10,0.58,0.29","67771,68013,68024,68492"
80609,4,1082,0.42,"287,239,280,275","0.44,0.39,0.47,0.37","67781,67782,67783,67784"
80610,4,1068,0.41,"246,377,334,110","0.30,0.48,0.41,0.41","68039,68041,68331,68478"


In [5]:
subsamp_dict = {}
for index, tileid in enumerate(explist['TILEID']):
    tileid_str = str(tileid)
    if tileid_str not in subsamp_dict.keys():
        subsamp_dict[tileid_str] = [list(map(int, explist[index]['EXPIDS'].split(',')))]
    else:
        subsamp_dict[tileid_str].append(list(map(int, explist[index]['EXPIDS'].split(','))))
        
subsamp_dict

{'80605': [[67972, 67974, 67987, 73704, 73705], [67973, 67975, 71594, 73702]],
 '80608': [[67771, 68013, 68491, 68492, 69625, 69626],
  [67771, 68013, 68024, 68492]],
 '80609': [[67781, 67782, 67783, 67784]],
 '80610': [[68039, 68041, 68331, 68478]]}

In [6]:
# exp_list = sorted([i for k in subsamp_dict.values() for j in k for i in j])
# print(len(exp_list), len(np.unique(exp_list)))

# mask = np.in1d(cat['EXPID'], exp_list)
# print(np.sum(mask))
# # cat[mask]

In [7]:
print('                 ELG_EFFTIME_DARK  BGS_EFFTIME_BRIGHT   EFFTIME_DARK_GFA   EFFTIME_BRIGHT_GFA')
print()

# for tileid in subsamp_dict.keys():
for tileid in ['80605', '80608', '80609', '80610']:
    mask = cat['TILEID']==int(tileid)
    print(cat['FAPRGRM'][mask][0].upper(), tileid)
    for subset_index, exp_list in enumerate(subsamp_dict[tileid]):
        # print(exp_list)
        mask = np.in1d(cat['EXPID'], exp_list)
        if np.sum(mask)>1:
            print_str = 'exposures'
        else:
            print_str = 'exposure '
        print('subset {}: {} {} {:11.0f} {:19.0f} {:18.0f} {:20.0f}'.format(subset_index+1, np.sum(mask), print_str, np.sum(cat['ELG_EFFTIME_DARK'][mask]), np.sum(cat['BGS_EFFTIME_BRIGHT'][mask]), np.sum(cat['EFFTIME_DARK_GFA'][mask]), np.sum(cat['EFFTIME_BRIGHT_GFA'][mask])))
    print()

                 ELG_EFFTIME_DARK  BGS_EFFTIME_BRIGHT   EFFTIME_DARK_GFA   EFFTIME_BRIGHT_GFA

LRGQSO 80605
subset 1: 5 exposures        1018                 968                  0                    0
subset 2: 4 exposures        1078                1078                  0                    0

ELG 80608
subset 1: 6 exposures        1110                1059                  0                    0
subset 2: 4 exposures        1114                1059                  0                    0

LRGQSO 80609
subset 1: 4 exposures        1094                1147                  0                    0

ELG 80610
subset 1: 4 exposures        1044                1063                  0                    0



In [8]:
# Print summary
for tileid_str in subsamp_dict.keys():
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    tile_flavor = cat['FAPRGRM'][mask][0].upper()
    n_exp = np.sum(mask)
    if tileid==80613:
        nom_depth = 180.
        depth_col = 'BGS_EFFTIME_BRIGHT'
    else:
        nom_depth = 1000.
        depth_col = 'EFFTIME_SPEC'
    tot_depth = np.sum(cat[depth_col][mask])
    print('Tile {} ({}):'.format(tileid, tile_flavor))
    print('all:       n_exp={:2}  {}={:5.0f}s'.format(n_exp, depth_col, tot_depth))
    subsets = subsamp_dict[tileid_str]
    for index, subset in enumerate(subsets):
        mask = np.in1d(cat['EXPID'], subset)
        subset_depth = np.sum(cat[depth_col][mask])
        print('subset {}:  n_exp={:2}  {}={:5.0f}s'.format(index+1, len(subset), depth_col, subset_depth))
    mask = (cat['TILEID']==tileid) & (~np.in1d(cat['EXPID'], np.concatenate(subsets)))
    unused_depth = np.sum(cat[depth_col][mask])
    print('unused:    n_exp={:2}  {}={:5.0f}s'.format(np.sum(mask), depth_col, unused_depth))
    print()

Tile 80605 (LRGQSO):
all:       n_exp=24  EFFTIME_SPEC= 6872s
subset 1:  n_exp= 5  EFFTIME_SPEC= 1018s
subset 2:  n_exp= 4  EFFTIME_SPEC= 1078s
unused:    n_exp=15  EFFTIME_SPEC= 4776s

Tile 80608 (ELG):
all:       n_exp=23  EFFTIME_SPEC=15160s
subset 1:  n_exp= 6  EFFTIME_SPEC= 1110s
subset 2:  n_exp= 4  EFFTIME_SPEC= 1114s
unused:    n_exp=16  EFFTIME_SPEC=13638s

Tile 80609 (LRGQSO):
all:       n_exp=15  EFFTIME_SPEC= 7819s
subset 1:  n_exp= 4  EFFTIME_SPEC= 1094s
unused:    n_exp=11  EFFTIME_SPEC= 6725s

Tile 80610 (ELG):
all:       n_exp=16  EFFTIME_SPEC= 9517s
subset 1:  n_exp= 4  EFFTIME_SPEC= 1044s
unused:    n_exp=12  EFFTIME_SPEC= 8472s

